# 00 — Setup, dataset & environment check

Verifies the working directory layout, the dataset export, and the Python/CUDA/Ultralytics
environment before running the pipeline (`01`–`06`).

> **Bring your own data.** This repo ships no images. Place your annotated dataset as
> `raw/export.zip` — a zip containing `images/*.jpg` and a single COCO
> `annotations_coco.json`. (`ROOT` below is the one working directory everything lives in;
> change it to your own path.)

## 1. Constants & folder structure

In [ ]:
import os
ROOT = "/home/jovyan/shared/s0598584"   # <- change to your working directory
for d in ["notebooks", "scripts", "raw", "crops_1280", "aug", "dataset"]:
    p = os.path.join(ROOT, d)
    print(f"{d:12s} -> {os.path.isdir(p)}")

## 2. Check the dataset `raw/export.zip` integrity

In [ ]:
import zipfile, os
zp = os.path.join(ROOT, "raw", "export.zip")
assert os.path.exists(zp), f"missing: {zp} (place your COCO export here)"
with zipfile.ZipFile(zp) as z:
    names = z.namelist()
    bad = z.testzip()
    exts = {}
    for n in names:
        e = os.path.splitext(n)[1].lower()
        exts[e] = exts.get(e, 0) + 1
print(f"export.zip: {os.path.getsize(zp)/1e6:.1f} MB, {len(names)} entries, testzip()={bad}")
print("extensions:", exts)

## 3. Inspect the COCO — derive classes from the export

In [ ]:
import json
with zipfile.ZipFile(zp) as z:
    with z.open("annotations_coco.json") as f:
        coco = json.load(f)
print("images      :", len(coco["images"]))
print("annotations :", len(coco["annotations"]))
print("categories  :", len(coco["categories"]))
for c in sorted(coco["categories"], key=lambda c: c["id"]):
    print(f"  {c['id']}: {c['name']}")

## 4. Verify the environment: PyTorch + CUDA + Ultralytics (YOLO26)

In [ ]:
import numpy as np, torch, ultralytics
print("numpy  :", np.__version__, "(must be < 2 for torch 2.1.2)")
print("torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")
print("ultralytics:", ultralytics.__version__)

## 5. Confirm scripts & pretrained init weights

In [ ]:
for s in ["image_crop_augment.py", "photometric_worker.py", "piheif_fix.py", "batch_finder.py"]:
    p = os.path.join(ROOT, "scripts", s)
    print(("[ok] " if os.path.exists(p) else "[--] ") + s)
w = os.path.join(ROOT, "yolo26n.pt")
print(("[ok] " if os.path.exists(w) else "[--] ") + "yolo26n.pt (official pretrained init)")

## 6. Disk budget

In [ ]:
import shutil
total, used, free = shutil.disk_usage(ROOT)
print(f"volume: total {total/1e9:.0f} GB | used {used/1e9:.0f} GB | free {free/1e9:.0f} GB")
assert free/1e9 > 60, "need >60 GB free for crops + augmentation"
print("disk budget OK.")

## ✅ Setup done

Continue with `01_crop.ipynb`.